# Polymorphism
- concept: same interface, different implementations
- static polymorphism: compile-time, method overloading
- dynamic polymorphism: run-time, method overriding

## The underlying mechanism of dynamic polymorphism
- virtual method table (vtable)
    - The compiler creates a vtable for each class with virtual functions

- Each object of the class contains a pointer to the vtable

## 1. Virtual functions + Polymorphism
- A virtual function is a member function that can be overridden in a derived class and allows for dynamic dispatch
- Runtime polymorphism is achieved through virtual functions, enabling the correct function to be called based on the actual object type at runtime
- The neccessary condition for virtual function call
    - The father class must declare the function as virtual
    - The signature of the function in the child class must be the same as that in the father class (controlled by the ``override`` keyword)
    - The function must be called through a pointer or reference to the child class

In [ ]:
%%writefile Sources/Polymorphism_1.cpp
#include <iostream>
using namespace std;

class Base {
public:
    virtual void show() {
        cout << "Base::show()" << endl;
    }
};

class Derived : public Base {
public:
    void show() override {
        cout << "Derived::show()" << endl;
    }
};

int main() {
    Derived d;
    Base* p = &d;
    Base& r = d;
    
    d.show();   // directly calls `Derived::show()`
    p->show();  // calls Derived::show() through base class pointer
    r.show();   // calls Derived::show() through base class reference

    Base b=d;  // object slicing occurs, only Base part is copied
    b.show();  // calls Base::show() due to object slicing
    // Derived d2=b; // error!!
    // d2.show(); // calls Base::show() due to object slicing
    return 0;
}


Overwriting Sources/Polymorphism_1.cpp


In [12]:
!g++ Sources/Polymorphism_1.cpp -o Polymorphism_1
!./Polymorphism_1

Derived::show()
Derived::show()
Derived::show()
Base::show()


> Slicing : only kept the part of the object that belongs to the base class, and the derived class part is sliced off. This can lead to unexpected behavior when calling virtual functions, as the base class version will be called instead of the derived class version.

## 2. Non-virtual functions without Polymorphism

In [11]:
%%writefile Sources/Polymorphism_2.cpp
#include <iostream>
using namespace std;

class Base {
public:
    void normal() {
        cout << "Base::normal()" << endl;
    }

    virtual void virt() {
        cout << "Base::virt()" << endl;
    }
};

class Derived : public Base {
public:
    void normal() {
        cout << "Derived::normal()" << endl;
    }

    void virt() override {
        cout << "Derived::virt()" << endl;
    }
};

int main() {
    Derived d;
    Base* p = &d;

    cout << "Non-virtual function:" << endl;
    p->normal();

    cout << "Virtual function:" << endl;
    p->virt();

    return 0;
}

Overwriting Sources/Polymorphism_2.cpp


In [12]:
!g++ Sources/Polymorphism_2.cpp -o Polymorphism_2
!./Polymorphism_2

Non-virtual function:
Base::normal()
Virtual function:
Derived::virt()


`normal()` is not declared as virtual in the base class, so it does not support polymorphism.

Which function is called depends on the type of the pointer/reference, not the actual object type. This is known as static binding or early binding.

**non-virtual functions: decided at compile time (depended by the type of the pointer/reference)**

**virtual functions: decided at runtime (depended by the actual type of the object)**

## 3. `override`: prevent 'fake overriding'

In [16]:
%%writefile Sources/Polymorphism_3.cpp
#include <iostream>
using namespace std;

class Base {
public:
    virtual void show() {
        cout << "Base::show()" << endl;
    }
};

class Derived : public Base {
public:
    void show(int x) {
        cout << "Derived::show(int)" << x << endl;
    }
};

int main() {
    Derived d;
    Base* p = &d;

    // d.show();   //error: no matching function for call to ‘Derived::show()’
    d.show(42);

    p->show();
    // p->show(42);  //error: no matching function for call to ‘Base::show(int)’
    

    return 0;
}

Overwriting Sources/Polymorphism_3.cpp


In [17]:
!g++ Sources/Polymorphism_3.cpp -o Polymorphism_3
!./Polymorphism_3

Derived::show(int)42
Base::show()


## 4. `const` contributes to the function signature

In [ ]:
%%writefile Sources/Polymorphism_4.cpp
#include <iostream>
using namespace std;

class Base {
public:
    virtual void show() const {
        cout << "Base::show() const" << endl;
    }
};

class Derived : public Base {
public:
    
    void show() {   // no const
        cout << "Derived::show()" << endl;
    }
};

int main() {
    Derived d;
    Base* p = &d;

    p->show();

    return 0;
}

Overwriting Sources/Polymorphism_4.cpp


In [25]:
!g++ Sources/Polymorphism_4.cpp -o Polymorphism_4
!./Polymorphism_4

Base::show() const


## 5. `final`: prevent further overrides in derived classes

In [26]:
%%writefile Sources/Polymorphism_5.cpp
#include <iostream>
using namespace std;

class Base {
public:
    virtual void show() {
        cout << "Base::show()" << endl;
    }
};

class Derived : public Base {
public:
    void show() override final {
        cout << "Derived::show()" << endl;
    }
};

// class Child : public Derived {
// public:
//     void show() override {   // 编译错误
//         cout << "Child::show()" << endl;
//     }
// };

int main() {
    Derived d;
    Base* p = &d;
    p->show();

    return 0;
}

Writing Sources/Polymorphism_5.cpp


In [27]:
!g++ Sources/Polymorphism_5.cpp -o Polymorphism_5
!./Polymorphism_5

Derived::show()


## 6. Pointers and Reference can trigger polymorphism, but not objects

In [28]:
%%writefile Sources/Polymorphism_6.cpp

#include <iostream>
using namespace std;

class Base {
public:
    virtual void show() {
        cout << "Base::show()" << endl;
    }
};

class Derived : public Base {
public:
    void show() override {
        cout << "Derived::show()" << endl;
    }
};

void testPointer(Base* p) {
    p->show();
}

void testReference(Base& r) {
    r.show();
}

int main() {
    Derived d;
    testPointer(&d);
    testReference(d);

    return 0;
}


Writing Sources/Polymorphism_6.cpp


In [29]:
!g++ Sources/Polymorphism_6.cpp -o Polymorphism_6
!./Polymorphism_6

Derived::show()
Derived::show()


## 7. Object slicing: objects do not trigger polymorphism

In [31]:
%%writefile Sources/Polymorphism_7.cpp

#include <iostream>
using namespace std;

class Base {
public:
    virtual void show() {
        cout << "Base::show()" << endl;
    }
};

class Derived : public Base {
public:
    void show() override {
        cout << "Derived::show()" << endl;
    }
};

int main() {
    Derived d;
    Base b = d;   // 对象切片

    b.show();

    return 0;
}

Writing Sources/Polymorphism_7.cpp


In [32]:
!g++ Sources/Polymorphism_7.cpp -o Polymorphism_7
!./Polymorphism_7

Base::show()


## 8. Force using the function from the Base class

In [33]:
%%writefile Sources/Polymorphism_8.cpp

#include <iostream>
using namespace std;

class Base {
public:
    virtual void show() {
        cout << "Base::show()" << endl;
    }
};

class Derived : public Base {
public:
    void show() override {
        cout << "Derived::show()" << endl;
    }

    void callBaseShow() {
        Base::show();   // 强制调用父类版本
    }
};

int main() {
    Derived d;
    d.show();
    d.callBaseShow();
    d.Base::show();

    return 0;
}

Writing Sources/Polymorphism_8.cpp


In [34]:
!g++ Sources/Polymorphism_8.cpp -o Polymorphism_8
!./Polymorphism_8

Derived::show()
Base::show()
Base::show()


## 9. Function Hideing

If the derived class defines a function with the same name as a function in the base class, this function will occupy the name in the function call.

In [39]:
%%writefile Sources/Polymorphism_9.cpp

#include <iostream>
using namespace std;

class Base {
public:
    virtual void show() {
        cout << "Base::show()" << endl;
    }

    void show(int x) {
        cout << "Base::show(int): " << x << endl;
    }
};

class Derived : public Base {
public:
    void show() override {
        cout << "Derived::show()" << endl;
    }
};

int main() {
    Derived d;
    d.show();
    // d.show(10);  // Compile error: Base::show(int) is hidden

    d.Base::show(10);  // Can explicitly call the base class version

    return 0;
}

Overwriting Sources/Polymorphism_9.cpp


In [40]:
!g++ Sources/Polymorphism_9.cpp -o Polymorphism_9
!./Polymorphism_9

Derived::show()
Base::show(int): 10


## 10. `using` to unhide the function from the base class


In [41]:
%%writefile Sources/Polymorphism_10.cpp
#include <iostream>
using namespace std;

class Base {
public:
    virtual void show() {
        cout << "Base::show()" << endl;
    }

    void show(int x) {
        cout << "Base::show(int): " << x << endl;
    }
};

class Derived : public Base {
public:
    using Base::show;   // 把父类同名重载重新引入

    void show() override {
        cout << "Derived::show()" << endl;
    }
};

int main() {
    Derived d;
    d.show();
    d.show(10);

    return 0;
}

Writing Sources/Polymorphism_10.cpp


In [42]:
!g++ Sources/Polymorphism_10.cpp -o Polymorphism_10
!./Polymorphism_10

Derived::show()
Base::show(int): 10


## 11. The final version for all 

In [43]:
%%writefile Sources/Polymorphism_11.cpp
#include <iostream>
using namespace std;

class Base {
public:
    virtual void show() {
        cout << "Base::show()" << endl;
    }

    virtual void showConst() const {
        cout << "Base::showConst() const" << endl;
    }

    void show(int x) {
        cout << "Base::show(int): " << x << endl;
    }
};

class Derived : public Base {
public:
    using Base::show;   // remain Base::show(int)

    void show() override {
        cout << "Derived::show()" << endl;
    }

    void showConst() const override final {
        cout << "Derived::showConst() const" << endl;
    }

    void callBaseShow() {
        Base::show();
    }
};

int main() {
    Derived d;
    Base* p = &d;
    Base& r = d;
    Base b = d;   // slicing

    cout << "1. object call directly：" << endl;
    d.show();

    cout << "2. base class pointer call (polymorphism)：" << endl;
    p->show();

    cout << "3. base class reference call (polymorphism)：" << endl;
    r.show();

    cout << "4. object slicing call：" << endl;
    b.show();

    cout << "5. call base class overloaded version：" << endl;
    d.show(100);

    cout << "6. force call base class version：" << endl;
    d.callBaseShow();
    d.Base::show();

    cout << "7. const virtual function：" << endl;
    p->showConst();

    return 0;
}

Writing Sources/Polymorphism_11.cpp


In [44]:
!g++ Sources/Polymorphism_11.cpp -o Polymorphism_11
!./Polymorphism_11

1. object call directly：
Derived::show()
2. base class pointer call (polymorphism)：
Derived::show()
3. base class reference call (polymorphism)：
Derived::show()
4. object slicing call：
Base::show()
5. call base class overloaded version：
Base::show(int): 100
6. force call base class version：
Base::show()
Base::show()
7. const virtual function：
Derived::showConst() const


## Summary

`Polymorphism`: call `show()` will get the different results due to the different actual object type.

`When Polymorphism happens`： 
- the function in the base class is virtual
- the derived class overrides the function in the base class
- the function is called through a pointer or reference to the base class

`override`: tell compiler "this function is intended to override a virtual function in the base class"". If there is no such function in the base class, the compiler will generate an error."

`final`: prevent further overriding of a virtual function in derived classes.

'overload vs override':
- `overload`: same function name, different parameters (number or type)
- `override`: same function signature (name and parameters), but in different classes (base and derived)
- `scope`: overload happens in the one class, while override happens in the base and derived classes